# Chunk 43 — Preprocess the 15×15 dataset for GATv2 fine-tuning

Same pipeline as Chunk 41 (12×12), retargeted to N = 15.

**Goal:** turn the ~7,000 new 15×15 `.mat` files into graphs the GATv2 models can train on, in exactly the format of the 25/35/45 data.
- **Graph builder:** the same frozen Chunk 5/12 code, copied verbatim — 4-neighbour edges, a virtual node, and 5 node features.
- **Raw-dB targets:** stored on disk; z-scored only at load time, with the canonical 25×25 statistics.
- **Seed mask:** built the same way as Chunk 1 (pixels that are metal in every file).
- **8-connected feed feature:** not stored. `backbone_no55` appends it at load time, as before.

**Checks before anything is written:**
- every file well-formed (15×15 binary pattern, 201 finite points, max ≤ 0 dB)
- the feed pixel predicted by `alternate.m` is always metal
- the builder reproduces 20 existing 35×35 graphs **bit-for-bit** from their raw files
- edge counts and patterns are verified on every graph
- duplicate patterns are found, and kept in the same split
- the normalisation round trip holds
- both GNNs run a finite forward pass on the 15×15 validation set

**Outputs.** All are new files; the chunk refuses to overwrite anything, and existing splits are untouched.

| file | what it is |
|---|---|
| `data/processed/processed_15x15.zip` | the graphs, staged by later chunks the same way as the other grids |
| `artifacts/seed_mask_15.npy` | the always-metal mask |
| `artifacts/ch43_manifest_15x15.csv` | one row per graph, including its source file |
| `splits/finetune15_{pool,val,test}_indices.json` | 80/10/10 splits, stratified on functioning |
| `artifacts/ch43_report.json` | checks, split sizes and file hashes |

Every output is written locally first, then copied to Drive, which is force-flushed and re-verified by SHA-256 (the Chunk 30 lesson).

## 1. Installs

In [1]:
!pip -q install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.8 MB/s eta 0:00:00


## 2. Repository and Drive
Clones or pulls the repo, puts `src/` on the path, and mounts Drive.

In [2]:
import os, sys, subprocess
REPO_ROOT = '/content/antenna_gnn'
if os.path.isdir(f'{REPO_ROOT}/.git'):
    subprocess.run(['git', '-C', REPO_ROOT, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', 'https://github.com/asparagusD/antenna_gnn.git', REPO_ROOT], check=True)
sys.path.insert(0, f'{REPO_ROOT}/src')
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA = '/content/drive/MyDrive/antenna_dataset'
ART, CKPT, SPLITS = f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints', f'{DATA_ROOT}/splits'
print(REPO_ROOT, DATA_ROOT)

Mounted at /content/drive
/content/antenna_gnn /content/drive/MyDrive/antenna_gnn


## 3. Locate the 15×15 files
**Before running:** the shared folder must be reachable from Colab. In Google Drive, right-click the folder and choose **Organize → Add shortcut → My Drive**. Then set `RAW_15` to its path, or leave it as `None` to search the usual places.

The cell stops if no files are found, or if two sub-folders contain the same file name.

In [3]:
# ── Locate the 15x15 .mat files ──
# The shared folder (Drive id 1v0z_S8SrUg9nEgo-3mvZ6-Vpau1jzoZp) must be reachable under /content/drive:
# in Drive, right-click the folder → "Organize" → "Add shortcut" → My Drive. Then either set RAW_15
# below, or leave it as None and the cell searches the usual places.
import glob
RAW_15 = None   # e.g. '/content/drive/MyDrive/15x15'  (set this if the search below does not find the files)
N = 15
CANDIDATES = [RAW_15] if RAW_15 else [f'{RAW_DATA}/fine-tuning dataset/15x15', f'{RAW_DATA}/15x15',
                                       '/content/drive/MyDrive/15x15', '/content/drive/MyDrive/antenna_dataset_15x15']
CANDIDATES += sorted(glob.glob('/content/drive/MyDrive/*15x15*')) + sorted(glob.glob('/content/drive/MyDrive/*/*15x15*'))
RAW_FILES = []
for c in CANDIDATES:
    if c and os.path.isdir(c):
        RAW_FILES = sorted(glob.glob(f'{c}/**/*.mat', recursive=True))
        if RAW_FILES:
            RAW_15 = c; break
assert RAW_FILES, ('STOP: no 15x15 .mat files found. Add the shared folder to My Drive (Add shortcut) and set RAW_15 '
                   f'to its path. Searched: {[c for c in CANDIDATES if c]}')
names = [os.path.basename(f) for f in RAW_FILES]
assert len(set(names)) == len(names), 'STOP: duplicate .mat file names in different sub-folders; resolve before preprocessing'
print(f'found {len(RAW_FILES):,} .mat files under {RAW_15}')
print('first / last:', names[0], '...', names[-1])

found 6,367 .mat files under /content/drive/MyDrive/antenna_dataset/fine-tuning dataset/15x15
first / last: Antenna_Seed_0.mat ... Antenna_Seed_6734.mat


## 4. Setup
**What it does:**
- Captures the mtimes of every existing artifact, split, checkpoint and processed zip, so the final guard can prove nothing was modified.
- **Stops if any 15×15 output already exists:** delete it deliberately if you want to rebuild.

In [4]:
# ── Setup ──
import json, hashlib, zipfile, shutil, concurrent.futures, math
from pathlib import Path
import numpy as np, pandas as pd, torch, scipy.io as sio
from scipy import ndimage
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm
from model import AntennaGNN
from feed_component import AppendFeedComponentFeature, feed_rc, feed_component_mask

def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
def OUT_ART(n):
    assert n.startswith('ch43_') or n == 'seed_mask_15.npy', n; return f'{ART}/{n}'
PRIOR = {p: os.path.getmtime(p) for pat in (f'{ART}/*', f'{SPLITS}/*', f'{CKPT}/*.pt', f'{DATA_ROOT}/data/processed/*.zip')
         for p in glob.glob(pat) if os.path.isfile(p)}
for p in (f'{ART}/seed_mask_15.npy', f'{SPLITS}/finetune15_pool_indices.json', f'{DATA_ROOT}/data/processed/processed_15x15.zip'):
    assert not os.path.exists(p), f'STOP: {p} already exists. Delete it deliberately to rebuild; this chunk never overwrites.'
PATCH_MM = 32.375                    # patch side used by Chunks 5/12 (pixel_size_mm = 32.375 / N)
LOCAL_RAW, LOCAL_PROC = '/content/raw_15x15', '/content/proc_15x15/15x15'
os.makedirs(LOCAL_RAW, exist_ok=True); os.makedirs(LOCAL_PROC, exist_ok=True)
print(f'captured mtimes of {len(PRIOR)} existing files; nothing will be overwritten')

captured mtimes of 223 existing files; nothing will be overwritten


## 5. Copy and inspect every file
**What it does:**
- Copies the `.mat` files to local SSD.
- Prints the fields of the first file. **Check here that the geometry matches the other grids:** a 32.4 mm patch, the same substrate and the same band.
- Checks every file: a 15×15 binary pattern and 201 finite S11 points no higher than 0 dB.
- Compares the two functioning labels: the legacy one (`resonant_freqs` non-empty) and `LABEL_MIN` (minimum < −10 dB).
- **New in this chunk:** for antennas where the labels disagree, it reports how many have their minimum at a band edge. On 25/35/45 the labels were identical; on 12×12 they differed on 5.3%. This tests the band-edge explanation.

In [5]:
# ── Copy to local SSD and inspect every file ──
def copy(src):
    dst = f'{LOCAL_RAW}/{os.path.basename(src)}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    return dst
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    LOCAL_FILES = list(tqdm(ex.map(copy, RAW_FILES), total=len(RAW_FILES), desc='copy .mat to local'))
first = sio.loadmat(LOCAL_FILES[0])
print('fields in the first file:', sorted(k for k in first if not k.startswith('__')))
geom = {k: np.asarray(first[k]).ravel()[:4] for k in first if not k.startswith('__') and k not in ('patch_pattern', 'S11_dB')}
print('other fields (first values):', {k: v.tolist() for k, v in geom.items()})

def inspect(path):
    m = sio.loadmat(path, variable_names=['patch_pattern', 'S11_dB', 'resonant_freqs'])
    p, s = np.asarray(m['patch_pattern']), np.asarray(m['S11_dB'], dtype=np.float64).ravel()
    return dict(file=os.path.basename(path), shape=p.shape, binary=bool(np.isin(p, (0, 1)).all()), n_metal=int(p.sum()),
                s11_len=s.size, finite=bool(np.isfinite(s).all()), s11_max=float(s.max()), s11_min=float(s.min()),
                n_resonances=int(np.asarray(m['resonant_freqs']).size), pattern=p.astype(np.uint8), s11=s)
with concurrent.futures.ThreadPoolExecutor(max_workers=16) as ex:
    REC = list(tqdm(ex.map(inspect, LOCAL_FILES), total=len(LOCAL_FILES), desc='inspect'))
bad = [r['file'] for r in REC if r['shape'] != (N, N) or not r['binary'] or r['s11_len'] != 201 or not r['finite']]
print(f'{len(bad)} malformed files' + (f': {bad[:10]}' if bad else ''))
assert not bad, 'STOP: malformed files (wrong pattern shape, non-binary pattern, S11 not 201 finite points)'
s11max = np.array([r['s11_max'] for r in REC]); s11min = np.array([r['s11_min'] for r in REC])
print(f'S11 range over all files: [{s11min.min():.2f}, {s11max.max():.4f}] dB')
assert s11max.max() <= 0.01, 'STOP: S11 above 0 dB: the curves are not raw |S11| in dB'
legacy = np.array([r['n_resonances'] > 0 for r in REC]); label_min = s11min < -10
print(f'functioning share: legacy (resonant_freqs non-empty) {legacy.mean():.3f} | LABEL_MIN (min < -10 dB) {label_min.mean():.3f} '
      f'| agreement {np.mean(legacy == label_min):.4f}')

# Where do the two labels disagree? On 25/35/45 they were identical; on 12x12 they differed on 5.3%.
dis = np.flatnonzero(legacy != label_min)
if len(dis):
    am = np.array([REC[i]['s11'].argmin() for i in dis])
    edge = (am <= 2) | (am >= 198)
    print(f'{len(dis)} disagreements: {edge.mean():.1%} have their minimum within 3 points of a band edge (1 or 4 GHz); '
          f'legacy=0 & min<-10 dB: {int(np.sum(~legacy[dis] & label_min[dis]))}, legacy=1 & min>=-10 dB: {int(np.sum(legacy[dis] & ~label_min[dis]))}')

copy .mat to local:   0%|          | 0/6367 [00:00<?, ?it/s]

fields in the first file: ['S11_dB', 'VSWR_full', 'patch_pattern', 'res_effs', 'res_gains', 'resonant_bws', 'resonant_freqs', 'resonant_vswrs']
other fields (first values): {'VSWR_full': [369.601746519613, 352.3284186884698, 336.24150914750294, 321.2434958052512], 'res_effs': [100], 'res_gains': [8.623009923784576], 'resonant_bws': [177.3781644799941], 'resonant_freqs': [3.76], 'resonant_vswrs': [1.2337714312618595]}


inspect:   0%|          | 0/6367 [00:00<?, ?it/s]

0 malformed files
S11 range over all files: [-43.26, -0.0104] dB
functioning share: legacy (resonant_freqs non-empty) 0.532 | LABEL_MIN (min < -10 dB) 0.556 | agreement 0.9760
153 disagreements: 100.0% have their minimum within 3 points of a band edge (1 or 4 GHz); legacy=0 & min<-10 dB: 153, legacy=1 & min>=-10 dB: 0


## 6. Seed (always-metal) mask
**What it does:**
- Computes the mask exactly as Chunk 1 did: pixels that are metal in every file.
- **Stops unless the feed pixel `alternate.m` predicts for N = 15 (row 3, column 7) is in it.** If it isn't, the 15×15 generator used a different geometry.
- Also checks that the expected centre block is contained in the mask, and compares the mask's size with the 25/35/45 masks.

In [6]:
# ── Seed (always-metal) mask, exactly as Chunk 1 built it, plus geometry checks ──
PAT = np.stack([r['pattern'] for r in REC])
SEED = np.all(PAT == 1, axis=0)
lab, ncomp = ndimage.label(SEED)
rows, cols = np.argwhere(SEED)[:, 0], np.argwhere(SEED)[:, 1]
print(f'always-metal pixels: {int(SEED.sum())} in {ncomp} block(s); rows {rows.min()}-{rows.max()}, cols {cols.min()}-{cols.max()}')
print('\n'.join(''.join('#' if v else '.' for v in row) for row in SEED))
fr, fc = feed_rc(N)
assert SEED[fr, fc], f'STOP: the feed pixel {(fr, fc)} predicted by alternate.m is not always metal; generator geometry differs'
# expected centre block from the generator: rows/cols round(0.35N)..round(0.65N) (MATLAB rounding, 1-based)
mr = lambda v: math.floor(v + 0.5)
lo, hi = mr(0.35 * N) - 1, mr(0.65 * N) - 1
centre = np.zeros((N, N), bool); centre[lo:hi + 1, lo:hi + 1] = True
print(f'expected centre block rows/cols {lo}-{hi}: contained in the always-metal set: {bool((SEED | ~centre).all())}')
# chance that a free pixel is metal in all files by luck is (fill)^n_files, negligible at this size
free_fill = PAT[:, ~SEED].mean()
print(f'mean fill of free pixels {free_fill:.3f}; P(a free pixel is metal in all {len(PAT):,} files by chance) ≈ {free_fill ** len(PAT):.1e}')
for M in (25, 35, 45):   # consistency with the existing grids' seed masks
    s = np.load(f'{ART}/seed_mask_{M}.npy'); print(f'  seed_mask_{M}: {int(s.sum())} pixels ({s.sum() / M**2:.1%} of the grid)')
print(f'  seed_mask_15: {int(SEED.sum())} pixels ({SEED.sum() / N**2:.1%} of the grid)')

always-metal pixels: 37 in 1 block(s); rows 3-9, cols 4-9
...............
...............
...............
.......#.......
....######.....
....######.....
....######.....
....######.....
....######.....
....######.....
...............
...............
...............
...............
...............
expected centre block rows/cols 4-9: contained in the always-metal set: True
mean fill of free pixels 0.549; P(a free pixel is metal in all 6,367 files by chance) ≈ 0.0e+00
  seed_mask_25: 65 pixels (10.4% of the grid)
  seed_mask_35: 145 pixels (11.8% of the grid)
  seed_mask_45: 197 pixels (9.7% of the grid)
  seed_mask_15: 37 pixels (16.4% of the grid)


## 7. Graph builder, verified
**What it does:**
- Copies `build_pyg_graph` verbatim from Chunks 5 and 12.
- **Reproduction test:** rebuilds 20 random existing 35×35 graphs from their raw `.mat` files and requires bit-identical `x`, `edge_index`, `edge_attr` and `y`. That proves the builder, the seed-mask convention and the file → `sample_i` numbering (sorted file order) are unchanged.
- The test is reported as SKIPPED only if the raw 35×35 files aren't on this Drive.

In [8]:
# ── Graph builder: VERBATIM from Chunk 5 / Chunk 12 (frozen), then verified against existing processed graphs ──
def build_pyg_graph(patch_pattern, s11_db, seed_mask, N):
    # Compute seed centroid
    coords = np.argwhere(seed_mask)
    seed_r, seed_c = coords.mean(axis=0)

    # Node features: (N*N + 1) nodes, 5 features each
    node_feats = []
    for i in range(N):
        for j in range(N):
            metal    = float(patch_pattern[i, j])
            x_norm   = j / (N - 1)
            y_norm   = i / (N - 1)
            is_seed  = float(seed_mask[i, j])
            dist_f   = np.sqrt((i - seed_r)**2 + (j - seed_c)**2) / N
            node_feats.append([metal, x_norm, y_norm, is_seed, dist_f])

    # Virtual global node (index N*N): all zeros except placeholder (is_seed=-1 for virtual node)
    node_feats.append([0.0, 0.5, 0.5, -1.0, 0.0])
    node_feats = torch.tensor(node_feats, dtype=torch.float)

    # 4-connectivity edges
    edge_src, edge_dst, edge_attr = [], [], []
    etype_map = {(1,1):0, (1,0):1, (0,1):2, (0,0):3}
    for i in range(N):
        for j in range(N):
            idx = i * N + j
            m_ij = int(patch_pattern[i, j])
            for (di, dj, direction) in [(0,1,0),(0,-1,1),(-1,0,2),(1,0,3)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < N and 0 <= nj < N:
                    nidx = ni * N + nj
                    m_nb = int(patch_pattern[ni, nj])
                    etype = etype_map[(m_ij, m_nb)]
                    edge_src.append(idx); edge_dst.append(nidx)
                    edge_attr.append([etype, direction])

    # Virtual node edges (connect to all metal pixels only)
    global_idx = N * N
    for i in range(N):
        for j in range(N):
            if patch_pattern[i, j] == 1:
                idx = i * N + j
                edge_src += [global_idx, idx]
                edge_dst += [idx, global_idx]
                edge_attr += [[4, 4], [4, 4]]  # virtual edge type

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    edge_attr  = torch.tensor(edge_attr, dtype=torch.float)

    # Target
    # RAW dB on disk — Dataset z-scores at load (see normalization contract)
    y = torch.tensor(s11_db, dtype=torch.float).unsqueeze(0)  # (1, 201)

    return Data(x=node_feats, edge_index=edge_index, edge_attr=edge_attr, y=y)

# Reproduction test: rebuild 20 existing 35x35 graphs from their raw .mat files (Chunk 12 numbered them by sorted
# file order) and require bit-identical tensors. Proves the builder, the seed-mask convention and the file -> index
# mapping are all unchanged. Skipped (with a warning) only if the raw 35x35 files are not on this Drive.
raw35 = sorted(glob.glob(f'{RAW_DATA}/fine-tuning dataset/35x35/**/Mat_Files/*.mat', recursive=True))
zip35 = f'{DATA_ROOT}/data/processed/processed_35x35.zip'
if raw35 and os.path.exists(zip35):
    seed35 = np.load(f'{ART}/seed_mask_35.npy')
    with zipfile.ZipFile(zip35) as Z:
        members = {os.path.basename(m): m for m in Z.namelist() if m.endswith('.pt')}
        picks = np.random.default_rng(41).choice(len(raw35), 20, replace=False)
        for i in tqdm(picks, desc='verifying 35x35 graphs'):
            m = sio.loadmat(raw35[i]); new = build_pyg_graph(m['patch_pattern'], m['S11_dB'].flatten(), seed35, 35)
            import io
            old = torch.load(io.BytesIO(Z.read(members[f'sample_{i}.pt'])), weights_only=False)
            for k in ('x', 'edge_index', 'edge_attr', 'y'):
                assert torch.equal(getattr(new, k), getattr(old, k)), f'STOP: rebuilt 35x35 sample_{i} differs in {k}'
    BUILDER_CHECK = 'bit-identical on 20 rebuilt 35x35 graphs'
else:
    BUILDER_CHECK = 'SKIPPED (raw 35x35 files or processed zip not found)'
print('builder reproduction check:', BUILDER_CHECK)

verifying 35x35 graphs:   0%|          | 0/20 [00:00<?, ?it/s]

builder reproduction check: bit-identical on 20 rebuilt 35x35 graphs


## 8. Build the 15×15 graphs
Writes `sample_<i>.pt`, where `i` is the position in the sorted file list. Each graph also carries `grid_size`, `pixel_size_mm` and the legacy `is_functioning` field.

**Checked on every graph:**
- 226 nodes, with the virtual node last
- raw-dB `y` of shape (1, 201)
- edge count = 4N(N−1) lattice edges + 2 × the number of metal pixels
- metal features equal to the pattern

In [9]:
# ── Build the 15x15 graphs (sample_<i>.pt, i = position in the sorted file list) ──
for i, r in enumerate(tqdm(REC, desc='build 15x15 graphs')):
    d = build_pyg_graph(r['pattern'], r['s11'], SEED, N)
    d.grid_size = N
    d.pixel_size_mm = PATCH_MM / N
    d.is_functioning = int(r['n_resonances'] > 0)          # legacy definition (Chunk 12)
    torch.save(d, f'{LOCAL_PROC}/sample_{i}.pt')
# structural checks on every graph
for i in tqdm(range(len(REC)), desc='verify graphs', leave=False):
    d = torch.load(f'{LOCAL_PROC}/sample_{i}.pt', weights_only=False)
    n_metal = REC[i]['n_metal']
    assert d.x.shape == (N * N + 1, 5) and float(d.x[-1, 3]) == -1.0
    assert d.y.shape == (1, 201) and float(d.y.max()) <= 0.01
    n_grid_edges = 4 * N * (N - 1)                          # directed 4-neighbour edges inside an N x N lattice
    assert d.edge_index.shape[1] == n_grid_edges + 2 * n_metal, f'sample_{i}: edge count'
    assert torch.equal(d.x[:-1, 0].reshape(N, N), torch.tensor(REC[i]['pattern'], dtype=torch.float))
print(f'built and verified {len(REC):,} graphs: {N*N+1} nodes each, 4-neighbour edges + virtual-node edges')

build 15x15 graphs:   0%|          | 0/6367 [00:00<?, ?it/s]

verify graphs:   0%|          | 0/6367 [00:00<?, ?it/s]

built and verified 6,367 graphs: 226 nodes each, 4-neighbour edges + virtual-node edges


## 9. Manifest and duplicates
**What it does:**
- Writes one manifest row per graph: source file, metal count, fill, minimum S11, both functioning labels and resonance count.
- Groups exactly repeated patterns. If any exist, their spread in minimum S11 measures how repeatable the simulation is — a direct estimate of label noise.

In [10]:
# ── Manifest and duplicate check ──
keys = [r['pattern'].tobytes() for r in REC]
first_seen, dup_of = {}, []
for i, k in enumerate(keys):
    dup_of.append(first_seen.setdefault(k, i))
dup_of = np.array(dup_of); n_dup = int((dup_of != np.arange(len(keys))).sum())
MAN = pd.DataFrame(dict(grid_size=N, sample_idx=np.arange(len(REC)), source_file=[r['file'] for r in REC],
                        n_metal=[r['n_metal'] for r in REC], fill=[r['n_metal'] / N**2 for r in REC],
                        min_s11_db=s11min, is_functioning=legacy.astype(int), label_min=label_min.astype(int),
                        n_resonances=[r['n_resonances'] for r in REC], duplicate_group=dup_of))
print(f'{n_dup} files repeat an earlier pattern exactly ({MAN.duplicate_group.nunique():,} distinct patterns)')
if n_dup:
    g = MAN[MAN.duplicate_group.duplicated(keep=False)].groupby('duplicate_group').min_s11_db.agg(['count', 'min', 'max'])
    print('max spread of min-S11 within a duplicate group (simulation repeatability):', f'{(g["max"] - g["min"]).max():.4f} dB')
print(MAN.describe().round(3).to_string())

1 files repeat an earlier pattern exactly (6,366 distinct patterns)
max spread of min-S11 within a duplicate group (simulation repeatability): 0.0000 dB
       grid_size  sample_idx   n_metal      fill  min_s11_db  is_functioning  label_min  n_resonances  duplicate_group
count     6367.0    6367.000  6367.000  6367.000    6367.000        6367.000   6367.000      6367.000         6367.000
mean        15.0    3183.000   140.235     0.623     -12.938           0.532      0.556         0.589         3182.216
std          0.0    1838.139    27.493     0.122       8.737           0.499      0.497         0.601         1838.432
min         15.0       0.000    87.000     0.387     -43.261           0.000      0.000         0.000            0.000
25%         15.0    1591.500   117.000     0.520     -18.895           0.000      0.000         0.000         1590.500
50%         15.0    3183.000   140.000     0.622     -11.441           1.000      1.000         1.000         3182.000
75%         15

## 10. Splits
**What it does:** an 80/10/10 pool/val/test split, matching Chunk 12's proportions.
- **Stratified** on `LABEL_MIN`, with seed 42.
- **Duplicate patterns stay in the same split,** so there is no train/test leakage.
- **Written to new `finetune15_*` files,** so every existing split is untouched.

In [11]:
# ── Splits: 80 / 10 / 10, stratified on functioning, duplicates kept together (Chunk 12 proportions) ──
# Written to NEW files (finetune15_*), so the existing finetune_* splits used by Chunks 30-40 are untouched.
from sklearn.model_selection import train_test_split
groups = MAN.groupby('duplicate_group').agg(label=('label_min', 'max'), members=('sample_idx', list)).reset_index()
g_tr, g_tmp = train_test_split(groups, test_size=0.20, stratify=groups.label, random_state=42)
g_va, g_te = train_test_split(g_tmp, test_size=0.50, stratify=g_tmp.label, random_state=42)
SPL15 = {name: sorted([[N, int(i)] for m in g.members for i in m]) for name, g in (('pool', g_tr), ('val', g_va), ('test', g_te))}
sets = {k: {tuple(v) for v in s} for k, s in SPL15.items()}
assert not (sets['pool'] & sets['val'] or sets['pool'] & sets['test'] or sets['val'] & sets['test']), 'splits overlap'
assert sum(len(s) for s in sets.values()) == len(MAN), 'splits do not cover the manifest'
for k, s in SPL15.items():
    idx = [i for _, i in s]
    print(f'{k:5s}: {len(s):5,d} antennas | functioning (LABEL_MIN) {MAN.label_min.iloc[idx].mean():.3f}')

pool : 5,092 antennas | functioning (LABEL_MIN) 0.556
val  :   638 antennas | functioning (LABEL_MIN) 0.556
test :   637 antennas | functioning (LABEL_MIN) 0.556


## 11. Ready-for-training checks
**What it does:**
- Checks the normalisation round trip under the `FinetuneDataset` contract, using the canonical 25×25 statistics.
- Checks that the 8-connected feed-component feature appends cleanly.
- **Smoke test:** runs zero-shot `best_model`, `backbone_no55` and (if present) the Chunk 42 four-grid model `ch42_ft12_replay_p100_s0` on the 15×15 validation set. The numbers are a baseline for the fine-tune and prove the graphs are model-ready. The test set is not touched.
- Prints the average-curve baseline for reference.

In [12]:
# ── Ready-for-training checks: normalisation round trip and a zero-shot forward pass on VAL ──
MEAN = torch.tensor(np.load(f'{ART}/s11_mean.npy'), dtype=torch.float32)   # canonical 25x25 statistics, read only
STD = torch.tensor(np.load(f'{ART}/s11_std.npy'), dtype=torch.float32)
def load15(i, normalise=True, feed_feature=False):
    d = torch.load(f'{LOCAL_PROC}/sample_{i}.pt', weights_only=False)
    d.y_raw = d.y.clone()
    if normalise:
        d.y = (d.y - MEAN) / (STD + 1e-8)                  # the FinetuneDataset contract (Chunk 12)
    if feed_feature:
        d = AppendFeedComponentFeature()(Data(x=d.x.clone(), edge_index=d.edge_index, edge_attr=d.edge_attr, y=d.y, y_raw=d.y_raw))
    return d
probe = [load15(i) for i in np.random.default_rng(42).choice(len(REC), 512, replace=False)]
y, yr = torch.cat([d.y for d in probe]), torch.cat([d.y_raw for d in probe])
rt = ((y * (STD + 1e-8) + MEAN) - yr).abs().max().item()
print(f'normalisation round trip max error {rt:.1e} dB; normalised y mean {y.mean():.3f}, std {y.std():.3f}')
assert rt < 1e-3
d6 = load15(0, feed_feature=True); assert d6.x.shape == (N * N + 1, 6)
print(f'feed-component feature (6th column) appends cleanly: {int(d6.x[:-1, 5].sum())} pixels in the feed component of sample_0')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
def gnn(nf):
    return AntennaGNN(node_feat_dim=nf, edge_feat_dim=2, hidden_dim=128, heads=8, edge_dim=16, num_blocks=4, output_dim=201)
SMOKE = {}
val_ids = [i for _, i in SPL15['val']]
REFS = [('best_model', 5, f'{CKPT}/best_model.pt'), ('backbone_no55', 6, f'{CKPT}/backbone_no55.pt')]
if os.path.exists(f'{CKPT}/ch42_ft12_replay_p100_s0.pt'):
    REFS.append(('ch42_ft12_replay (12/25/35/45)', 6, f'{CKPT}/ch42_ft12_replay_p100_s0.pt'))
for name, nf, path in REFS:
    ck = torch.load(path, map_location='cpu', weights_only=False)
    m = gnn(nf); m.load_state_dict(ck['model_state'] if 'model_state' in ck else ck, strict=True); m = m.to(DEVICE).eval()
    P, T = [], []
    with torch.no_grad():
        for b in DataLoader([load15(i, feed_feature=(nf == 6)) for i in val_ids], batch_size=128):
            b = b.to(DEVICE); P.append((m(b) * STD.to(DEVICE) + MEAN.to(DEVICE)).cpu()); T.append(b.y_raw.view(-1, 201).cpu())
    P, T = torch.cat(P), torch.cat(T)
    assert torch.isfinite(P).all()
    SMOKE[name] = dict(val_mse_db2=float(((P - T) ** 2).mean()), val_mae_db=float((P - T).abs().mean()))
    print(f'{name:32s} ZERO-SHOT on 15x15 VAL ({len(val_ids)} antennas): MSE {SMOKE[name]["val_mse_db2"]:.3f} dB², '
          f'MAE {SMOKE[name]["val_mae_db"]:.3f} dB   (finite; the graphs are model-ready)')
print('Reference: average-curve baseline on 15x15 VAL:',
      f'{float(((T - torch.tensor(np.stack([REC[i]["s11"] for _, i in SPL15["pool"]]).mean(0), dtype=torch.float32)) ** 2).mean()):.3f} dB²')

normalisation round trip max error 3.8e-06 dB; normalised y mean -0.187, std 0.808
feed-component feature (6th column) appends cleanly: 171 pixels in the feed component of sample_0
best_model                       ZERO-SHOT on 15x15 VAL (638 antennas): MSE 4.843 dB², MAE 0.818 dB   (finite; the graphs are model-ready)
backbone_no55                    ZERO-SHOT on 15x15 VAL (638 antennas): MSE 3.231 dB², MAE 0.676 dB   (finite; the graphs are model-ready)
ch42_ft12_replay (12/25/35/45)   ZERO-SHOT on 15x15 VAL (638 antennas): MSE 2.109 dB², MAE 0.512 dB   (finite; the graphs are model-ready)
Reference: average-curve baseline on 15x15 VAL: 5.165 dB²


## 12. Write durably
**What it does:**
- Zips the graphs and writes the seed mask, manifest, splits and report locally.
- Copies them to Drive. It never overwrites an existing file.
- Force-flushes Drive, remounts, and verifies every file by SHA-256.

In [13]:
# ── Write outputs durably (local first, then Drive, forced flush, re-verify) ──
STAGE = '/content/ch43_out'; os.makedirs(STAGE, exist_ok=True)
zip_local = f'{STAGE}/processed_15x15.zip'
with zipfile.ZipFile(zip_local, 'w', compression=zipfile.ZIP_STORED) as Z:
    for i in tqdm(range(len(REC)), desc='zip'):
        Z.write(f'{LOCAL_PROC}/sample_{i}.pt', arcname=f'15x15/sample_{i}.pt')
np.save(f'{STAGE}/seed_mask_15.npy', SEED)
MAN.to_csv(f'{STAGE}/ch43_manifest_15x15.csv', index=False)
for k, s in SPL15.items():
    json.dump(s, open(f'{STAGE}/finetune15_{k}_indices.json', 'w'))
DEST = {'processed_15x15.zip': f'{DATA_ROOT}/data/processed/processed_15x15.zip', 'seed_mask_15.npy': f'{ART}/seed_mask_15.npy',
        'ch43_manifest_15x15.csv': f'{ART}/ch43_manifest_15x15.csv',
        **{f'finetune15_{k}_indices.json': f'{SPLITS}/finetune15_{k}_indices.json' for k in SPL15}}
HASH = {n: sha(f'{STAGE}/{n}') for n in DEST}
REPORT = dict(n_files=len(REC), raw_folder=RAW_15, grid=N, pixel_size_mm=PATCH_MM / N, seed_pixels=int(SEED.sum()),
              feed_rc=list(feed_rc(N)), builder_check=BUILDER_CHECK, duplicates=n_dup,
              functioning_legacy=float(legacy.mean()), functioning_label_min=float(label_min.mean()),
              label_agreement=float(np.mean(legacy == label_min)), splits={k: len(v) for k, v in SPL15.items()},
              zero_shot_val=SMOKE, files={n: dict(path=p, sha256=HASH[n], bytes=os.path.getsize(f'{STAGE}/{n}')) for n, p in DEST.items()},
              note='Graphs use the frozen Chunk 5/12 builder (4-neighbour edges + virtual node). The 8-connected feed-'
                   'component feature is NOT stored; backbone_no55 appends it at load time with AppendFeedComponentFeature.')
json.dump(REPORT, open(f'{STAGE}/ch43_report.json', 'w'), indent=2)
DEST['ch43_report.json'] = f'{ART}/ch43_report.json'; HASH['ch43_report.json'] = sha(f'{STAGE}/ch43_report.json')
for n, p in DEST.items():
    assert not os.path.exists(p), f'STOP: {p} exists; refusing to overwrite'
    os.makedirs(os.path.dirname(p), exist_ok=True); shutil.copy(f'{STAGE}/{n}', p)
print('flushing Drive...')
drive.flush_and_unmount(); drive.mount('/content/drive', force_remount=True)
for n, p in DEST.items():
    assert sha(p) == HASH[n], f'STOP: {n} did not persist on Drive'
print('all outputs verified on Drive after a forced flush:')
for n, p in DEST.items(): print(f'  {p}  ({os.path.getsize(p)/1e6:.1f} MB)')

zip:   0%|          | 0/6367 [00:00<?, ?it/s]

flushing Drive...
Mounted at /content/drive
all outputs verified on Drive after a forced flush:
  /content/drive/MyDrive/antenna_gnn/data/processed/processed_15x15.zip  (222.7 MB)
  /content/drive/MyDrive/antenna_gnn/artifacts/seed_mask_15.npy  (0.0 MB)
  /content/drive/MyDrive/antenna_gnn/artifacts/ch43_manifest_15x15.csv  (0.5 MB)
  /content/drive/MyDrive/antenna_gnn/splits/finetune15_pool_indices.json  (0.1 MB)
  /content/drive/MyDrive/antenna_gnn/splits/finetune15_val_indices.json  (0.0 MB)
  /content/drive/MyDrive/antenna_gnn/splits/finetune15_test_indices.json  (0.0 MB)
  /content/drive/MyDrive/antenna_gnn/artifacts/ch43_report.json  (0.0 MB)


## 13. Guards
**What it checks:**
- no existing file was modified
- all outputs are present on Drive
- the zip holds exactly one graph per `.mat` file
- the splits cover every graph exactly once
- the feed pixel is in the always-metal set

In [14]:
# ── Guards ──
fails = []
def guard(ok, msg):
    print(f"  [{'PASS' if ok else 'FAIL'}] {msg}")
    if not ok: fails.append(msg)
guard(all(os.path.getmtime(p) == t for p, t in PRIOR.items() if os.path.exists(p)), f'no existing file modified ({len(PRIOR)})')
guard(all(os.path.exists(p) for p in DEST.values()), 'all outputs present on Drive')
with zipfile.ZipFile(DEST['processed_15x15.zip']) as Z:
    n_pt = sum(m.endswith('.pt') for m in Z.namelist())
guard(n_pt == len(REC), f'zip holds {n_pt:,} graphs = number of .mat files')
guard(sum(len(v) for v in SPL15.values()) == len(REC), 'splits cover every graph exactly once')
guard(SEED[feed_rc(N)], 'feed pixel is in the always-metal set')
assert not fails, fails
print('ALL GUARDS PASSED. 15x15 data is ready for GATv2 fine-tuning.')

  [PASS] no existing file modified (223)
  [PASS] all outputs present on Drive
  [PASS] zip holds 6,367 graphs = number of .mat files
  [PASS] splits cover every graph exactly once
  [PASS] feed pixel is in the always-metal set
ALL GUARDS PASSED. 15x15 data is ready for GATv2 fine-tuning.
